# Mining Data from Dynamic Webpage

In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager

from bs4 import BeautifulSoup
import pandas as pd
import time

# -----------------------------
# Setup Selenium Chrome driver
# -----------------------------
options = Options()

# Run without opening a browser window
options.add_argument("--headless=new")

# Prevent some resource issues on macOS/Linux
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--no-sandbox")

# Automatically download correct ChromeDriver version
driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()),
    options=options
)

# -----------------------------
# Open website
# -----------------------------
url = "https://www.asicminervalue.com/efficiency"

driver.get(url)

# Give page time to load
time.sleep(3)

# -----------------------------
# Scroll until all rows load
# -----------------------------
last_height = driver.execute_script(
    "return document.body.scrollHeight"
)

while True:
    # Scroll to bottom
    driver.execute_script(
        "window.scrollTo(0, document.body.scrollHeight);"
    )

    # Wait for new content to load
    time.sleep(2)

    # Get new page height
    new_height = driver.execute_script(
        "return document.body.scrollHeight"
    )

    # Stop if no new content appears
    if new_height == last_height:
        break

    last_height = new_height

# -----------------------------
# Parse page source
# -----------------------------
html = driver.page_source

driver.quit()

soup = BeautifulSoup(html, "html.parser")

# Find table
table = soup.find(
    "table",
    {"class": "w-full caption-bottom text-sm"}
)

if table is None:
    print("Table not found.")
    exit()

# -----------------------------
# Extract headers
# -----------------------------
headers = [
    th.get_text(strip=True)
    for th in table.find_all("th")
]

# -----------------------------
# Extract rows
# -----------------------------
data = []

rows = table.find_all("tr")[1:]

print(f"Rows found: {len(rows)}")

for row in rows:
    cols = row.find_all("td")

    row_data = [
        col.get_text(" ", strip=True)
        for col in cols
    ]

    if row_data:
        data.append(row_data)

# -----------------------------
# Create DataFrame
# -----------------------------
df = pd.DataFrame(data, columns=headers)

# -----------------------------
# Save CSV
# -----------------------------
output_file = "data/asic_miner_ASIC_hardware.csv"

df.to_csv(output_file, index=False)

print(f"Saved {len(df)} rows to {output_file}")

/Users/pj/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Rows found: 248
Saved 248 rows to data/asic_miner_ASIC_hardware____delete.csv


## JSON for bitcoin hashrate
- Hashrate data from Blockchain.com: https://www.blockchain.com/explorer/charts/hash-rate

In [ ]:
import json
import pandas as pd

# -----------------------------
# Load JSON file
# -----------------------------
# Hashrate data from Blockchain.com: https://www.blockchain.com/explorer/charts/hash-rate
with open('data/hash-rate_30DayAverage_AllYears.json', 'r') as f:
    raw = json.load(f)

# -----------------------------
# Extract hashrate data
# -----------------------------
hashrate = pd.DataFrame(raw['hash-rate'])

hashrate.rename(columns={
    'x': 'timestamp',
    'y': 'hashrate[TH/s]'
}, inplace=True)

# -----------------------------
# Extract market price data
# -----------------------------
price = pd.DataFrame(raw['market-price'])

price.rename(columns={
    'x': 'timestamp',
    'y': 'price[USD]'
}, inplace=True)

# -----------------------------
# Merge on timestamp
# -----------------------------
data = pd.merge(
    hashrate,
    price,
    on='timestamp',
    how='outer'
)

# -----------------------------
# Convert timestamps
# -----------------------------
data['datetime'] = pd.to_datetime(
    data['timestamp'],
    unit='ms'
)

# Optional: reorder columns
data = data[
    [
        'timestamp',
        'datetime',
        'hashrate[TH/s]',
        'price[USD]'
    ]
]

# -----------------------------
# Save CSV
# -----------------------------
output_path = 'data/bitcoin-hashrate.csv'

data.to_csv(output_path, index=False)

print(f"Saved {len(data)} rows to {output_path}")

Saved 1623 rows to data/bitcoin-hashrate___delete.csv
